Production Ready Pipeline covering file loading, API ingestion, and quality validation



In [ ]:
import pandas as pd
import requests
from pathlib import Path

# Load data with explicit parameters
def load_csv_safe(path, date_cols=None, dtypes=None):
  """Load CSV with all safeguards against silent corruption."""
  return pd.read_csv(
      path,
      encoding='utf-8',
      dtype=dtypes or {},
      parse_dates=date_cols or [],
      na_values=['','NULL','N/A', 'null', 'None', '-', '--', 'n/a'],
      low_memory=False
  )

#Example usage
df = load_csv_safe(
    'messy_ecommerce_customers_10k.csv',
    date_cols=['signup_date','last_login_date','last_purchase_date'],
    dtypes={'customer_id': str, 'first_name': str}
)


# 2. Five Quality Audit Check

def quality_audit(df, name='Dataset'):
  """Run the 5 essential checks on any new dataset."""
  print(f"\n{'='*50}")
  print(f"Quality Audit: {name}")
  print(f"{'='*50}")

  #Check Shape
  print(f"\n1.Shape: {df.shape[0]:,} rows x {df.shape[1]:,} cols")

  #Check Missing Values
  missing = df.isnull().sum()
  missing_pct = (missing / len(df) * 100).round(1)
  has_missing = missing[missing > 0]
  if len(has_missing) > 0:
    print(f"\n2.Missing Values ({len(has_missing)} columns):")
    for col in has_missing.index:
      print(f" {col}: {has_missing[col]:,} ({missing_pct[col]}%)")
  else:
    print(f"\n2. Missing values: None!")

  # 3. Data types
  print(f"\n3.Data types")
  for dtype, count in df.dtypes.value_counts().items():
    print(f" {dtype}: {count} columns")

  # 4. Check Duplicates
  n_dupes = df.duplicated().sum()
  print(f"\n4. Duplicates: {n_dupes:,} ({n_dupes/len(df) * 100}:.1f%)")

  # 5. Value ranges (numerical only)
  print(f"\n5.Value ranges (numeric)")
  for col in df.select_dtypes(include='number').columns[:5]:
    print(f" {col}: [{df[col].min():.2f}, {df[col].max():.2f}]")

quality_audit(df, "Customer Data")



Quality Audit: Customer Data

1.Shape: 10,050 rows x 18 cols

2.Missing Values (4 columns):
 age: 499 (5.0%)
 gender: 303 (3.0%)
 preferred_category: 203 (2.0%)
 customer_satisfaction_score: 805 (8.0%)

3.Data types
 object: 11 columns
 float64: 4 columns
 datetime64[ns]: 2 columns
 bool: 1 columns

4. Duplicates: 50 (0.4975124378109453:.1f%)

5.Value ranges (numeric)
 age: [-48.00, 299.00]
 total_orders: [0.00, 149.00]
 customer_satisfaction_score: [1.00, 14.00]
 cart_abandonment_rate: [0.00, 1.00]
